In [1]:
!pip install kagglehub
!pip install langdetect
!pip install transformers
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=2a7a9792170f90b9b1f06fba381b73e41f052b0b16019cd3e450f9abb10d47da
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 4.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12

In [68]:
# Disable W&B logging for clean ouput
import os
os.environ["WANDB_DISABLED"] = "true"
import kagglehub
import numpy as np
import pandas as pd
from langdetect import detect
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer,BertForSequenceClassification, BertModel, Trainer, TrainingArguments
# from torch.utils.data import Dataset
from datasets import DatasetDict, Dataset
from sklearn.metrics import accuracy_score
import torch.nn as nn
import torch
from sklearn.utils.class_weight import compute_class_weight

In [5]:
# Download dataset
def load_dataset(used_column):
    dataset_path = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")
    print("dataset downloaded to this path:", dataset_path)
    ds = pd.read_csv(os.path.join(dataset_path,'aa_dataset-tickets-multi-lang-5-2-50-version.csv'),usecols=used_column)
    ds = ds.rename(columns={'queue': 'label'})
    return ds

In [6]:
# Retain necessary columns
used_column = ['body','queue'] #['type','queue','priority']
ds = load_dataset(used_column)

# Check null label value, and discard the value
if ds['label'].isnull().any():
    print('There are some rows that has null label value, discard the rows')
    ds = ds.dropna(subset=['label'])

# We use the langdetect library to label the language used in the text
ds['lang'] = ds['body'].apply(lambda x: detect(str(x)))

dataset downloaded to this path: /kaggle/input/multilingual-customer-support-tickets


In [7]:
# Discard non english text
ds = ds[ds['lang'] == 'en']
ds = ds.drop(columns=['lang']).reset_index(drop=True)

# Display labels
print('Unique value for category:', ds['label'].unique())
# Check label distribution
print('Label freq:', ds['label'].value_counts(normalize=True) * 100)
# Display total row number for each label
print('Label freq:', ds['label'].value_counts())

# Enumerate categories
label2id = {label: idx for idx, label in enumerate(ds['label'].unique())}
ds['label'] = ds['label'].map(label2id)

# reverse
# id2label = {v: k for k, v in label2id.items()}

Unique value for category: ['Technical Support' 'Returns and Exchanges' 'Billing and Payments'
 'Sales and Pre-Sales' 'Service Outages and Maintenance' 'Product Support'
 'IT Support' 'Customer Service' 'Human Resources' 'General Inquiry']
Label freq: label
Technical Support                  29.400061
Product Support                    18.513417
Customer Service                   14.615856
IT Support                         12.049791
Billing and Payments                9.769411
Returns and Exchanges               5.106622
Service Outages and Maintenance     4.060810
Sales and Pre-Sales                 3.045608
Human Resources                     2.050811
General Inquiry                     1.387614
Name: proportion, dtype: float64
Label freq: label
Technical Support                  5763
Product Support                    3629
Customer Service                   2865
IT Support                         2362
Billing and Payments               1915
Returns and Exchanges              1001
S

In [61]:
ds_ready = ds.copy()
# Split the dataset
train_ds, test_ds = train_test_split(ds_ready, test_size=0.2, random_state=42, stratify=ds_ready['label'])
# Convert to Huggingface dataset object
train_dataset = Dataset.from_pandas(train_ds.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_ds.reset_index(drop=True))
ready_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

In [62]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(datas):
    return tokenizer(datas["body"], padding="max_length", truncation=True, max_length=114)

tokenized_dataset = ready_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/15681 [00:00<?, ? examples/s]

Map:   0%|          | 0/3921 [00:00<?, ? examples/s]

In [69]:
num_labels = train_ds['label'].nunique()
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# # Get class weights based on label column
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(train_ds['label']),
#     y=train_ds['label']
# )

# # Convert to tensor for PyTorch
# class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# class WeightedDistilBERT(DistilBertForSequenceClassification):
#     def __init__(self, config, class_weights):
#         super().__init__(config)
#         self.class_weights = class_weights

#     def compute_loss(self, model, inputs, return_outputs=False):
#         labels = inputs.get("labels")
#         outputs = model(**inputs)
#         logits = outputs.get("logits")
#         loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
#         loss = loss_fct(logits, labels)
#         return (loss, outputs) if return_outputs else loss

# # Config with correct number of labels
# config = DistilBertConfig.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

# # Initialize model with custom weights
# model = WeightedDistilBERT(config, class_weights_tensor)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [70]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)  # Get class with max logit
    labels = pred.label_ids
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'macro_f1': f1,
        'macro_precision': precision,
        'macro_recall': recall
    }

In [71]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    learning_rate=1e-5, 
    logging_dir="./logs",
    logging_steps=10,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 